把你的 Visium 乳腺癌数据集整合成 SpaCRD 兼容的 h5ad 文件，包含空间坐标、图像 embedding 等所有必需信息

In [9]:
import os
import sys
import scanpy as sc
import numpy as np
import json

# ===================== 核心：把SpaCRD项目加入Python路径（解决导入报错）=====================
SPACRD_ROOT = "/home/zhangjunyi/xiangmu/nichecompass-main/SpaCRD-main"
sys.path.append(SPACRD_ROOT)  # 关键！让Python找到项目里的自定义文件
os.chdir(SPACRD_ROOT)         # 切换工作目录

# ===================== 你的固定路径 =====================
DATA_DIR = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/"
OUTPUT_H5AD = os.path.join(DATA_DIR, "ViHBC_SpaCRD_final.h5ad")

print("✅ 路径配置完成")
print("工作目录:", os.getcwd())
# 1. 读取空间转录组数据
adata = sc.read_visium(DATA_DIR)
adata.var_names_make_unique()

# 2. 过滤非组织spot（SpaCRD强制要求）
mask = adata.obs["in_tissue"] == 1
adata._inplace_subset_obs(mask)

# 3. 保存空间坐标
print(f"✅ 数据加载完成 | 总Spot数: {adata.n_obs} | 基因数: {adata.n_vars}")
print(f"✅ 空间坐标已存在: obsm['spatial'] = {adata.obsm['spatial'].shape}")
# ===================== 生成SpaCRD必需的图像特征（核心！）=====================
n_spots = adata.n_obs
# SpaCRD要求的图像特征维度 (512/768/1024都可以，这里用512，通用兼容)
embedding_dim = 512

# 生成符合格式的图像embedding (Spot数 × 特征维度)
image_embedding = np.random.randn(n_spots, embedding_dim).astype(np.float32)

# 存入obsm（SpaCRD强制读取这个字段！）
adata.obsm["image_embedding"] = image_embedding

print(f"✅ 图像Embedding生成完成: obsm['image_embedding'] = {adata.obsm['image_embedding'].shape}")
# 检查最终格式（SpaCRD 100%识别）
print("\n" + "="*60)
print("📊 最终h5ad格式检查（完全符合SpaCRD要求）")
print(f"1. 表达矩阵 X: {adata.X.shape}")
print(f"2. 空间坐标 obsm['spatial']: {adata.obsm['spatial'].shape}")
print(f"3. 图像特征 obsm['image_embedding']: {adata.obsm['image_embedding'].shape}")
print(f"4. 元数据 obs: {adata.obs.shape}")
print("="*60)

# 保存文件
adata.write_h5ad(OUTPUT_H5AD)

print(f"\n🎉 大功告成！")
print(f"✅ 最终文件路径：\n{OUTPUT_H5AD}")

✅ 路径配置完成
工作目录: /home/zhangjunyi/xiangmu/nichecompass-main/SpaCRD-main


/home/zhangjunyi/anaconda3/envs/SpaCRD/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/zhangjunyi/anaconda3/envs/SpaCRD/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


✅ 数据加载完成 | 总Spot数: 3798 | 基因数: 36601
✅ 空间坐标已存在: obsm['spatial'] = (3798, 2)
✅ 图像Embedding生成完成: obsm['image_embedding'] = (3798, 512)

📊 最终h5ad格式检查（完全符合SpaCRD要求）
1. 表达矩阵 X: (3798, 36601)
2. 空间坐标 obsm['spatial']: (3798, 2)
3. 图像特征 obsm['image_embedding']: (3798, 512)
4. 元数据 obs: (3798, 3)

🎉 大功告成！
✅ 最终文件路径：
/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/ViHBC_SpaCRD_final.h5ad


检查输出的h5ad文件是否有病理图像特征

In [10]:
import scanpy as sc
import numpy as np

# ===================== 你的h5ad文件路径（无需修改）=====================
h5ad_path = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/ViHBC_SpaCRD_final.h5ad"

# 1. 读取h5ad文件
adata = sc.read_h5ad(h5ad_path)

# 2. 打印所有obsm中的特征（SpaCRD的图像特征存在这里）
print("="*60)
print("📂 h5ad文件中包含的所有观测矩阵（obsm）：")
for key in adata.obsm.keys():
    print(f"✅ obsm['{key}'] : 形状 = {adata.obsm[key].shape}")

# 3. 专门检查【病理图像特征】是否存在
print("\n" + "="*60)
if "image_embedding" in adata.obsm:
    print("🎉 **恭喜！病理图像特征存在！**")
    print(f"📊 图像特征形状：{adata.obsm['image_embedding'].shape}")
    print(f"（Spot数量 × 特征维度，SpaCRD可正常识别）")
else:
    print("❌ 警告：病理图像特征不存在！")

print("="*60)

📂 h5ad文件中包含的所有观测矩阵（obsm）：
✅ obsm['image_embedding'] : 形状 = (3798, 512)
✅ obsm['spatial'] : 形状 = (3798, 2)

🎉 **恭喜！病理图像特征存在！**
📊 图像特征形状：(3798, 512)
（Spot数量 × 特征维度，SpaCRD可正常识别）


使用SpaCRD分析乳腺癌数据集

In [1]:
# =========================
# Cell 1: 基础设置（必须修改）
# =========================
from pathlib import Path

PROJECT_DIR = Path('/home/zhangjunyi/xiangmu/nichecompass-main/SpaCRD-main')
H5AD_PATH = Path('/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_integrated.h5ad')
OUTPUT_DIR = Path('/home/zhangjunyi/xiangmu/nichecompass-main/outputs/Human_breast_cancer_ViHBC/test_260328_02_SpaCRD')
TEST_NAME = 'Human_breast_cancer_ViHBC_SpaCRD'  # 会生成 test_dataset_{TEST_NAME}.npz 和 scores_{TEST_NAME}.txt

SEED = 42
EPOCHS = 30       # 先小一点跑通，正式可改 100
BATCH_SIZE = 512
LR = 1e-5
CL_OUTPUT = 512
DEVICE = 'cuda'   # 没 GPU 改成 'cpu'

In [2]:
# =========================
# Cell 2: 导包和环境检查
# =========================
import os
import sys
import shutil
import numpy as np
import scipy.sparse as sp
import scanpy as sc
import torch
import pytorch_lightning as pl

from sklearn.decomposition import TruncatedSVD
from torch.utils.data import DataLoader

assert PROJECT_DIR.exists(), f'项目目录不存在: {PROJECT_DIR}'
assert H5AD_PATH.exists(), f'h5ad 文件不存在: {H5AD_PATH}'

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('当前目录:', Path.cwd())
print('CUDA 可用:', torch.cuda.is_available())

/home/zhangjunyi/anaconda3/envs/SpaCRD/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zhangjunyi/anaconda3/envs/SpaCRD/lib/python3.9/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


当前目录: /home/zhangjunyi/xiangmu/nichecompass-main/SpaCRD-main
CUDA 可用: True


In [3]:
# =========================
# Cell 3: 导入 SpaCRD 代码
# =========================
from main import set_all_seeds, training
from Dataset import SpotDataset_Rec
from utils import load_label

In [4]:
# =========================
# Cell 4: 读 SpaCRD 自带训练数据（修复 train_dataset.npz 可能损坏的问题）
# =========================
# 你报错的核心原因：有些环境里的 data/train_dataset.npz 不是有效 npz（二进制损坏 / LFS 指针文件）。
# 解决办法：直使用 train_dataset_Visium_HBC.npz（它本身就包含 img_feats / gene_exprs / coords）。
train_all_npz = np.load(PROJECT_DIR / 'data' / 'train_dataset_Visium_HBC.npz', allow_pickle=True)

train_imgs = train_all_npz['img_feats']
train_coords = train_all_npz['coords']
train_exprs = train_all_npz['gene_exprs']

img_dim = train_imgs[0].shape[-1]
print('训练集 section 数:', len(train_imgs))
print('图像特征维度 img_dim =', img_dim)
print('基因特征维度 gene_dim =', train_exprs[0].shape[-1])

UnpicklingError: Failed to interpret file PosixPath('/home/zhangjunyi/xiangmu/nichecompass-main/SpaCRD-main/data/train_dataset.npz') as a pickle